# 01 — Databricks GenAI Environment

## Objective

Get oriented in the Databricks environment and the basic building blocks you will reuse in every later notebook: notebooks themselves, mixing Python and SQL, DataFrames, Unity Catalog concepts, Delta tables, and a conceptual (non-executed) overview of the AI capabilities this project will build on.

This notebook does **not** build an application. It is a orientation lap around the environment.

## What We Will Learn

- How a Databricks notebook mixes Python cells, SQL cells (`%sql`), and markdown
- How Unity Catalog organizes data: `catalog.schema.table`
- What a Delta table is and why Databricks uses it by default
- How to create, write, and query a small table end to end
- What `dbutils.widgets` are for and why we use them instead of hardcoding names
- A conceptual map of the AI Functions (`ai_query`, `ai_classify`, `ai_extract`, `ai_parse_document`) and Model Serving that later notebooks will use

## Prerequisites

- A Databricks workspace with Unity Catalog enabled
- A cluster or SQL warehouse attached to this notebook
- `USE CATALOG` / `CREATE SCHEMA` privileges on at least one catalog (ask your workspace admin if you're unsure which catalog to use — `main` is the common default on trial/dev workspaces)
- Basic familiarity with Python; no prior Databricks experience assumed

## Conceptual Explanation

**Workspace organization.** A Databricks workspace holds notebooks, clusters (compute for interactive/batch work), SQL warehouses (compute for SQL/BI workloads), jobs, and the Unity Catalog metastore. This notebook only needs a cluster or a SQL warehouse — nothing else.

**Notebooks.** A notebook is a sequence of cells. Each cell has a language (Python, SQL, Scala, R, or Markdown). In a Python notebook you can run a SQL cell inline with the `%sql` magic, or run SQL from Python via `spark.sql("...")`. Both are used below so you can see the difference.

**DataFrames.** A Spark DataFrame is a distributed, tabular in-memory structure — conceptually like a pandas DataFrame, but it can scale across a cluster. Almost everything in this project (documents, chunks, embeddings, evaluation results) will eventually live in a DataFrame or a Delta table.

**Unity Catalog.** Unity Catalog is Databricks' governance layer for data. Every table is addressed as `catalog.schema.table` (a three-level namespace), and access is controlled centrally rather than per-cluster. You will set your own `catalog`/`schema` via widgets below rather than us hardcoding names that may not exist in your workspace.

**Delta tables.** Delta Lake is the default table format in Databricks: Parquet files plus a transaction log, giving you ACID writes, schema enforcement, and time travel (you can query a table *as of* a previous version). When you `saveAsTable(...)` without specifying a format, Databricks uses Delta.

**AI capability overview (conceptual only — not executed here).** Later notebooks will use Databricks AI Functions, which are SQL/Python functions that call an LLM per-row inside a query:

| Function | Purpose | Used in |
|---|---|---|
| `ai_parse_document` | Extract text/structure from PDFs and other documents | Notebook 04 |
| `ai_classify` | Classify text into categories | Notebook 05 |
| `ai_extract` | Pull structured fields out of text | Notebook 06 |
| `ai_query` | Call any Model Serving endpoint (e.g. an LLM) from SQL/Python | RAG notebooks (10-12) |

These all depend on **Model Serving** (managed LLM endpoints) being available in your workspace/region, which varies by account. We are *not* calling them in this notebook — we're only naming them so the map is in your head before you meet them for real.

## Example Data

A tiny, fully synthetic dataset: 5 rows describing fictional documents from a fictional bank, "Fictional bank" (no real institution). Just enough rows to prove a DataFrame → Delta table round trip; later notebooks build a larger synthetic catalog.

In [ ]:
# Small synthetic dataset -- fictional bank, fictional documents only.
sample_documents = [
    (1, "Checking Account Overview",         "Product",           "Retail Banking"),
    (2, "Wire Transfer Procedure",            "Operations",        "Payments Ops"),
    (3, "KYC Policy Summary",                 "Compliance",        "Compliance"),
    (4, "Disputing a Card Charge",            "Customer Service",  "Contact Center"),
    (5, "Core Banking API Overview",          "Technical",         "Engineering"),
]

columns = ["doc_id", "title", "category", "department"]

df = spark.createDataFrame(sample_documents, columns)
display(df)

## Implementation

### Step 1 — Parameterize catalog and schema

We use `dbutils.widgets` instead of a hardcoded catalog/schema name. Widgets create a small input box at the top of the notebook, so you can point this notebook at a catalog/schema that actually exists in **your** workspace without editing code. Change the default values below if `main` isn't available to you.

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema (created if missing)")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
table_fqn = f"{catalog_name}.{schema_name}.sample_documents"

print(f"Target table: {table_fqn}")

### Step 2 — Create the schema (only if you have `CREATE SCHEMA` privileges on this catalog)

If this fails with a permissions error, ask your workspace admin for a catalog/schema you can write to, and re-run the widget cell above with those values.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

### Step 3 — Write the DataFrame as a Delta table

No format is specified, so Databricks defaults to Delta.

In [ ]:
df.write.mode("overwrite").saveAsTable(table_fqn)
print(f"Wrote {df.count()} rows to {table_fqn}")

### Step 4 — Query it back with SQL

This uses the `%sql` magic, Databricks' shorthand for running a SQL cell inside a Python notebook. Note it can't reference the Python variable `table_fqn` directly -- magics work on literal text -- so the fully qualified name below must match the widget values you set in Step 1.

In [ ]:
%sql
-- Update catalog/schema below to match the widgets in Step 1 if you changed them
SELECT * FROM main.genai_lab.sample_documents ORDER BY doc_id

### Step 5 — Read it back into Python

This is the pattern you'll use constantly: `spark.table(...)` loads a Delta table as a DataFrame, so Python and SQL cells can freely hand data back and forth through the same table.

In [ ]:
roundtrip_df = spark.table(table_fqn)
display(roundtrip_df.filter(roundtrip_df.category == "Compliance"))

## Inspect the Output

- In the `display()` outputs above, check that all 5 rows round-tripped correctly and column types look right (`doc_id` should show as a number, not a string).
- Run `DESCRIBE HISTORY` below to see the transaction log entry Delta created for your write -- this is what powers time travel and audit trails.
- Open the Unity Catalog Explorer (left sidebar in Databricks) and find `catalog_name.schema_name.sample_documents` to see the table's schema, lineage tab, and sample data in the UI.

In [ ]:
%sql
DESCRIBE HISTORY main.genai_lab.sample_documents

## Experimentation Section

Try these before moving on:

1. Change the widget values to a different catalog/schema you have access to and re-run everything.
2. Add 2-3 more fictional documents to `sample_documents` and re-run the write with `mode("append")` instead of `"overwrite")`. Then check `DESCRIBE HISTORY` again -- how many versions do you see now?
3. Run `SELECT * FROM <table> VERSION AS OF 0` to time-travel back to the first version.
4. Change a column type (e.g. make `doc_id` a string) and re-run the write -- does Delta's schema enforcement let it through silently, or complain?
5. In the Catalog Explorer UI, look at the table's **Lineage** tab -- it's currently empty. Keep this in mind; it will start filling in once notebooks read *from* this table into other tables.

## Common Errors / Limitations

- **`CATALOG_DOES_NOT_EXIST` / permission denied on `CREATE SCHEMA`** -- the `catalog_name` widget points at a catalog you can't write to. Ask an admin which catalog/schema to use, or ask for one to be created for you.
- **`%sql` cell doesn't see your Python variables** -- magics operate on the literal text in the cell. If you change the widget defaults, you must also edit the SQL cells' fully qualified table names by hand (or use `spark.sql(f"...")` from Python instead of `%sql`).
- **No cluster/warehouse attached** -- notebooks need compute attached before any cell will run; attach one from the notebook toolbar.
- **AI Functions and Model Serving are not exercised here** -- their availability depends on workspace region/tier and is not guaranteed. We only introduced them conceptually; notebooks 04-06 and 10-12 are where you'll find out if they're enabled for you.

## Summary

You created a DataFrame from Python literals, wrote it as a Delta table under a Unity Catalog `catalog.schema.table` path you control via widgets, read it back through both SQL and Python, and inspected its transaction history. Every later notebook in this project reuses exactly this loop: **Python/SQL → DataFrame → Delta table → query back**. You also now have a name-level map of the AI Functions (`ai_parse_document`, `ai_classify`, `ai_extract`, `ai_query`) that later notebooks will actually call.

## Suggested Exercises

- Create a second small table (any fictional data you like) in the same schema and query both together with a `JOIN`.
- Try `DESCRIBE EXTENDED main.genai_lab.sample_documents` and compare the extra metadata it shows versus `DESCRIBE HISTORY`.
- Read the [Unity Catalog docs](https://docs.databricks.com/) overview page for the "three-level namespace" concept and see how it maps to what you just did.
- When you're ready, move on to **`02_synthetic_data_generation.ipynb`**, which builds the larger fictional document + CSV dataset that the rest of this project reuses.